In [ ]:
import os
import time
import random
import numpy as np
import json
import argparse
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import dgl
from dgl.dataloading import GraphDataLoader
from tqdm import tqdm
from ogb.graphproppred import DglGraphPropPredDataset, Evaluator
from ogb.graphproppred.mol_encoder import AtomEncoder, BondEncoder

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
def compute_avg_degree(dataset):
    log_degrees = []

    for g, _ in dataset:
        d = g.in_degrees().float()
        log_degrees.append(torch.log(d + 1.0))

    all_log_degrees = torch.cat(log_degrees)
    avg_log_d = all_log_degrees.mean().item()

    return avg_log_d

In [ ]:
class FCLayer(nn.Module):
    def __init__(self, in_size, out_size, activation='relu'):
        super().__init__()
        self.linear = nn.Linear(in_size, out_size)

        if activation.lower() == 'relu':
            self.act = nn.ReLU()
        elif activation.lower() == 'leakyrelu':
            self.act = nn.LeakyReLU()
        else:
            self.act = nn.Identity()

    def forward(self, x):
        return self.act(self.linear(x))

class MLP(nn.Module):
    def __init__(self, in_size, hidden_size, out_size, layers, mid_activation='relu'):
        super().__init__()
        self.net = nn.ModuleList()
        if layers == 1:
            self.net.append(nn.Linear(in_size, out_size))
        else:
            self.net.append(FCLayer(in_size, hidden_size, mid_activation))
            for _ in range(layers - 2):
                self.net.append(FCLayer(hidden_size, hidden_size, mid_activation))
            self.net.append(nn.Linear(hidden_size, out_size))

    def forward(self, x):
        for layer in self.net:
            x = layer(x)
        return x

In [ ]:
class PNATower(nn.Module):
    def __init__(self, in_features, out_features, aggregators, scalers, avg_log_d,
                 pretrans_layers=1, posttrans_layers=1, edge_dim=0,
                 use_batch_norm=True, use_graph_norm=True, dropout=0.0):
        super().__init__()
        self.aggregators = aggregators
        self.scalers = scalers
        self.avg_log_d = avg_log_d
        self.use_edge_feat = edge_dim > 0

        pretrans_in_dim = 2 * in_features + edge_dim
        self.pretrans = MLP(in_size=pretrans_in_dim, hidden_size=in_features,
                            out_size=in_features, layers=pretrans_layers)

        aggr_dim = len(aggregators) * len(scalers) * in_features
        self.posttrans = MLP(in_size=in_features + aggr_dim, hidden_size=out_features,
                             out_size=out_features, layers=posttrans_layers)

        self.use_graph_norm = use_graph_norm
        self.use_batch_norm = use_batch_norm
        if use_batch_norm:
            self.batch_norm = nn.BatchNorm1d(out_features)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, g, h, snorm_n, e=None):
        with g.local_scope():
            g.ndata['h'] = h
            if self.use_edge_feat and e is not None:
                g.edata['e'] = e

            def message_func(edges):
                if self.use_edge_feat and e is not None:
                    h_cat = torch.cat([edges.src['h'], edges.dst['h'], edges.data['e']], dim=-1)
                else:
                    h_cat = torch.cat([edges.src['h'], edges.dst['h']], dim=-1)
                return {'msg': self.pretrans(h_cat)}

            def reduce_func(nodes):
                msg = nodes.mailbox['msg']
                aggr_results = []
                for aggr in self.aggregators:
                    if aggr == 'sum':
                        aggr_results.append(torch.sum(msg, dim=1))
                    elif aggr == 'max':
                        aggr_results.append(torch.max(msg, dim=1)[0])
                    elif aggr == 'min':
                        aggr_results.append(torch.min(msg, dim=1)[0])
                    elif aggr == 'mean':
                        aggr_results.append(torch.mean(msg, dim=1))
                    elif aggr == 'std':
                        var = torch.var(msg, dim=1, unbiased=False)
                        aggr_results.append(torch.sqrt(var + 1e-5))
                return {'m_aggr': torch.cat(aggr_results, dim=-1)}

            g.update_all(message_func, reduce_func)
            m_aggr = g.ndata['m_aggr']

            if self.scalers == ['identity']:
                m_scaled = m_aggr
            else:
                deg = g.in_degrees().float().unsqueeze(-1).clamp(min=1)

                scale_results = []
                for scaler in self.scalers:
                    if scaler == 'identity':
                        scale_results.append(m_aggr)
                    elif scaler == 'amplification':
                        scale_results.append(m_aggr * (torch.log(deg + 1.0) / self.avg_log_d))
                    elif scaler == 'attenuation':
                        scale_results.append(m_aggr * (self.avg_log_d / torch.log(deg + 1.0)))
                m_scaled = torch.cat(scale_results, dim=-1)

            out = self.posttrans(torch.cat([h, m_scaled], dim=-1))

            if self.use_graph_norm and snorm_n is not None:
                out = out * snorm_n

            if self.use_batch_norm:
                out = self.batch_norm(out)

            out = self.dropout(out)

            return out

In [ ]:
class PNALayer(nn.Module):
    def __init__(self, in_features, out_features, aggregators, scalers, avg_log_d, towers=5,
                 pretrans_layers=1, posttrans_layers=1, divide_input=True, residual=True,
                 edge_dim=0, use_batch_norm=True, use_graph_norm=True, dropout=0.0):
        super().__init__()
        self.towers_list = nn.ModuleList()
        self.divide_input = divide_input
        self.residual = residual

        self.input_tower = in_features // towers if divide_input else in_features
        self.output_tower = out_features // towers

        for _ in range(towers):
            self.towers_list.append(
                PNATower(self.input_tower, self.output_tower, aggregators, scalers, avg_log_d,
                         pretrans_layers, posttrans_layers, edge_dim,
                         use_batch_norm, use_graph_norm, dropout)
            )

        self.mixing_network = FCLayer(out_features, out_features, activation='LeakyReLU')

    def forward(self, g, h, snorm_n, e=None):
        h_in = h
        if self.divide_input:
            tower_outs = []
            for i, tower in enumerate(self.towers_list):
                h_chunk = h[:, i * self.input_tower : (i + 1) * self.input_tower]
                tower_outs.append(tower(g, h_chunk, snorm_n, e))
            y = torch.cat(tower_outs, dim=-1)
        else:
            y = torch.cat([tower(g, h, snorm_n, e) for tower in self.towers_list], dim=-1)

        y = self.mixing_network(y)

        if self.residual and h_in.shape == y.shape:
            y = y + h_in

        return y

In [ ]:
class RealWorldBenchmarkModel(nn.Module):
    def __init__(self, config, avg_log_d=None):
        super().__init__()
        self.edge_feat = config.edge_feat

        self.in_feat_dropout = nn.Dropout(config.in_feat_dropout)


        if config.name == 'ZINC':
            self.node_encoder = nn.Embedding(num_embeddings=28, embedding_dim=config.hidden_dim)
            if self.edge_feat:
                self.edge_encoder = nn.Embedding(num_embeddings=4, embedding_dim=config.edge_dim)
                pass_edge_dim = config.edge_dim
            else:
                pass_edge_dim = 0
        elif config.name == 'MolHIV':
            self.node_encoder = AtomEncoder(emb_dim=config.hidden_dim)
            if self.edge_feat:
                self.edge_encoder = BondEncoder(emb_dim=config.edge_dim)
                pass_edge_dim = config.edge_dim
            else:
                pass_edge_dim = 0
        else:
            self.node_encoder = nn.Linear(config.node_dim, config.hidden_dim)
            if self.edge_feat:
                self.edge_encoder = nn.Linear(1, config.edge_dim)
                pass_edge_dim = config.edge_dim
            else:
                pass_edge_dim = 0

        self.layers = nn.ModuleList()


        for i in range(config.num_layers):
            is_first = (i == 0)
            is_last = (i == config.num_layers - 1)

            if is_first: divide_in = config.divide_input_first
            elif is_last: divide_in = config.divide_input_last
            else: divide_in = True

            layer_in_dim = config.hidden_dim
            layer_out_dim = config.out_dim if is_last else config.hidden_dim

            self.layers.append(
                PNALayer(layer_in_dim, layer_out_dim, config.aggregators, config.scalers, avg_log_d,
                         towers=config.towers, pretrans_layers=config.pretrans_layers,
                         posttrans_layers=config.posttrans_layers, divide_input=divide_in,
                         residual=True, edge_dim=pass_edge_dim,
                         use_batch_norm=config.use_batch_norm, use_graph_norm=config.use_graph_norm,
                         dropout=config.dropout)
            )


        self.graph_decoder = nn.Sequential(
            nn.Linear(config.out_dim, config.out_dim // 2),
            nn.ReLU(),
            nn.Linear(config.out_dim // 2, config.task_dim)
        )

    def forward(self, g, h, snorm_n, e=None):

        h = self.node_encoder(h)
        h = self.in_feat_dropout(h)


        if self.edge_feat and e is not None:
            e = self.edge_encoder(e)


        for layer in self.layers:
            h = layer(g, h, snorm_n, e)


        g.ndata['h_out'] = h
        h_graph = dgl.readout_nodes(g, 'h_out', op='sum')


        out = self.graph_decoder(h_graph)
        return out

In [ ]:
def train_epoch(model, optimizer, dataloader, device, config):
    model.train()
    epoch_loss = 0.0
    epoch_metric = 0.0

    for batched_graph, labels in dataloader:
        batched_graph, h, e, labels = config.preprocess(
            batched_graph, labels, device, config.edge_feat
        )

        tab_sizes = batched_graph.batch_num_nodes()
        snorm_n = [torch.ones(size, 1) / math.sqrt(size) for size in tab_sizes]
        snorm_n = torch.cat(snorm_n).to(device)

        pred = model(batched_graph, h, snorm_n, e)

        loss = config.loss_fn(pred, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_size = batched_graph.batch_size
        epoch_loss += loss.item() * batch_size
        epoch_metric += config.calc_metric(pred, labels, batch_size)

    num_samples = len(dataloader.dataset)
    return epoch_loss / num_samples, epoch_metric / num_samples


def evaluate_epoch(model, dataloader, device, config):

    model.eval()
    epoch_loss = 0.0
    epoch_metric = 0.0

    with torch.no_grad():
        for batched_graph, labels in dataloader:
            batched_graph, h, e, labels = config.preprocess(
                batched_graph, labels, device, config.edge_feat
            )

            tab_sizes = batched_graph.batch_num_nodes()
            snorm_n = [torch.ones(size, 1) / math.sqrt(size) for size in tab_sizes]
            snorm_n = torch.cat(snorm_n).to(device)

            pred = model(batched_graph, h, snorm_n, e)

            loss = config.loss_fn(pred, labels)

            batch_size = batched_graph.batch_size
            epoch_loss += loss.item() * batch_size
            epoch_metric += config.calc_metric(pred, labels, batch_size)

    num_samples = len(dataloader.dataset)
    return epoch_loss / num_samples, epoch_metric / num_samples


def make_graph_data_bundle(train_dataset, val_dataset, test_dataset,
                           batch_size=128, drop_last_train=False,
                           num_workers=2, pin_memory=True):
    """Build consistently named datasets/loaders and the training degree statistic."""
    datasets = {
        'train': train_dataset,
        'valid': val_dataset,
        'test': test_dataset,
    }
    loaders = {
        split: GraphDataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=(split == 'train'),
            drop_last=(drop_last_train if split == 'train' else False),
            num_workers=num_workers,
            pin_memory=pin_memory,
        )
        for split, dataset in datasets.items()
    }
    return {
        'datasets': datasets,
        'loaders': loaders,
        'avg_log_d': compute_avg_degree(train_dataset),
    }


def make_superpixel_data_bundle(dataset_class, train_size,
                                batch_size=128, seed=42,
                                drop_last_train=True, num_workers=2,
                                pin_memory=True):
    """Load a superpixel benchmark and create its deterministic train/valid split."""
    full_train_dataset = dataset_class(split='train')
    test_dataset = dataset_class(split='test')
    valid_size = len(full_train_dataset) - train_size
    split_generator = torch.Generator().manual_seed(seed)
    train_dataset, val_dataset = torch.utils.data.random_split(
        full_train_dataset,
        [train_size, valid_size],
        generator=split_generator,
    )
    return make_graph_data_bundle(
        train_dataset,
        val_dataset,
        test_dataset,
        batch_size=batch_size,
        drop_last_train=drop_last_train,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )


def make_ogb_data_bundle(dataset_name, root, batch_size=128,
                         num_workers=2, pin_memory=True):
    """Load an OGB graph benchmark using its official split indices."""
    ogb_dataset = DglGraphPropPredDataset(name=dataset_name, root=root)
    split_indices = ogb_dataset.get_idx_split()
    train_dataset = torch.utils.data.Subset(ogb_dataset, split_indices['train'])
    val_dataset = torch.utils.data.Subset(ogb_dataset, split_indices['valid'])
    test_dataset = torch.utils.data.Subset(ogb_dataset, split_indices['test'])
    return make_graph_data_bundle(
        train_dataset,
        val_dataset,
        test_dataset,
        batch_size=batch_size,
        drop_last_train=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )


def run_benchmark_experiment(config_factory, data_bundle, preprocess_fn,
                             loss_fn, metric_fn, device, checkpoint_path,
                             experiment_name, epochs=None):
    """Train one ZINC/image model; return its best-validation summary."""
    experiment_config = config_factory()
    if epochs is not None:
        experiment_config.epochs = epochs
    experiment_config.preprocess = preprocess_fn
    experiment_config.loss_fn = loss_fn
    experiment_config.calc_metric = metric_fn

    experiment_model = RealWorldBenchmarkModel(
        experiment_config, data_bundle['avg_log_d']
    ).to(device)
    experiment_optimizer = optim.Adam(
        experiment_model.parameters(),
        lr=experiment_config.init_lr,
        weight_decay=experiment_config.weight_decay,
    )
    experiment_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        experiment_optimizer,
        mode=experiment_config.scheduler_mode,
        factor=experiment_config.lr_reduce_factor,
        patience=experiment_config.lr_schedule_patience,
    )

    maximize_metric = experiment_config.scheduler_mode == 'max'
    best_val_metric = float('-inf') if maximize_metric else float('inf')
    best_test_metric = float('nan')
    loaders = data_bundle['loaders']

    for epoch_index in range(experiment_config.epochs):
        train_loss, train_metric = train_epoch(
            experiment_model, experiment_optimizer, loaders['train'],
            device, experiment_config,
        )
        val_loss, val_metric = evaluate_epoch(
            experiment_model, loaders['valid'], device, experiment_config,
        )
        experiment_scheduler.step(val_metric)
        current_lr = experiment_optimizer.param_groups[0]['lr']

        improved = (val_metric > best_val_metric if maximize_metric
                    else val_metric < best_val_metric)
        test_text = ''
        if improved:
            best_val_metric = val_metric
            _, best_test_metric = evaluate_epoch(
                experiment_model, loaders['test'], device, experiment_config,
            )
            torch.save(experiment_model.state_dict(), checkpoint_path)
            test_text = f' | Test: {best_test_metric:.4f} (New Best!)'

        print(
            f'{experiment_name} | Epoch {epoch_index:03d} | '
            f'LR: {current_lr:.6f} | Train: {train_loss:.4f} | '
            f'Val: {val_metric:.4f}{test_text}'
        )
        if current_lr < experiment_config.min_lr:
            break

    print(f'{experiment_name} | Best validation: {best_val_metric:.4f} | '
          f'Test at best validation: {best_test_metric:.4f}')
    return {
        'best_val_metric': best_val_metric,
        'test_metric_at_best_val': best_test_metric,
        'checkpoint_path': checkpoint_path,
    }


In [ ]:
class BaseConfig:
    epochs = 1000
    batch_size = 128
    init_lr = 0.001
    lr_reduce_factor = 0.5
    min_lr = 1e-5
    weight_decay = 3e-6

    num_layers = 4
    towers = 5
    pretrans_layers = 1
    posttrans_layers = 1
    use_batch_norm = True
    use_graph_norm = True
    in_feat_dropout = 0.0


# 1. ZINC
class BaseZINCConfig(BaseConfig):
    name = 'ZINC'
    task_dim = 1
    node_dim = 28
    scheduler_mode = 'min'

# --- 1.1 ZINC PNA ---
class ZINC_PNA_NOEDGE(BaseZINCConfig):
    aggregators = ['mean', 'max', 'min', 'std']
    scalers = ['identity', 'amplification', 'attenuation']
    edge_feat = False
    edge_dim = 0
    hidden_dim = 75
    out_dim = 70
    dropout = 0.0
    divide_input_first = False
    divide_input_last = True
    lr_schedule_patience = 5

class ZINC_PNA_EDGE(BaseZINCConfig):
    aggregators = ['mean', 'max', 'min', 'std']
    scalers = ['identity', 'amplification', 'attenuation']
    edge_feat = True
    edge_dim = 50
    hidden_dim = 70
    out_dim = 60
    dropout = 0.0
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 20

# --- 1.2 ZINC PNA (No Scalers) ---
class ZINC_PNA_NOSCALER_NOEDGE(BaseZINCConfig):
    aggregators = ['mean', 'max', 'min', 'std']
    scalers = ['identity']
    edge_feat = False
    edge_dim = 0
    hidden_dim = 95
    out_dim = 90
    dropout = 0.0
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5

class ZINC_PNA_NOSCALER_EDGE(BaseZINCConfig):
    aggregators = ['mean', 'max', 'min', 'std']
    scalers = ['identity']
    edge_feat = True
    edge_dim = 50
    hidden_dim = 90
    out_dim = 80
    dropout = 0.0
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 20

# --- 1.3 ZINC MPNN (Sum) ---
class ZINC_MPNN_SUM_NOEDGE(BaseZINCConfig):
    aggregators = ['sum']
    scalers = ['identity']
    edge_feat = False
    edge_dim = 0
    hidden_dim = 110
    out_dim = 80
    dropout = 0.0
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5
    weight_decay = 1e-5

class ZINC_MPNN_SUM_EDGE(BaseZINCConfig):
    aggregators = ['sum']
    scalers = ['identity']
    edge_feat = True
    edge_dim = 50
    hidden_dim = 100
    out_dim = 70
    dropout = 0.0
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 20

# --- 1.4 ZINC MPNN (Max) ---
class ZINC_MPNN_MAX_NOEDGE(BaseZINCConfig):
    aggregators = ['max']
    scalers = ['identity']
    edge_feat = False
    edge_dim = 0
    hidden_dim = 110
    out_dim = 80
    dropout = 0.0
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5
    weight_decay = 1e-5

class ZINC_MPNN_MAX_EDGE(BaseZINCConfig):
    aggregators = ['max']
    scalers = ['identity']
    edge_feat = True
    edge_dim = 50
    hidden_dim = 100
    out_dim = 70
    dropout = 0.0
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 20

def preprocess_zinc(batched_graph, labels, device, use_edge_feat=True):
    batched_graph = batched_graph.to(device)
    labels = labels.to(device).float()
    h = batched_graph.ndata['feat'].long().squeeze()

    if use_edge_feat and batched_graph.num_edges() > 0:
        e = batched_graph.edata['feat'].long().squeeze()
    else:
        e = None
    return batched_graph, h, e, labels

def zinc_loss(pred, labels):
    return F.l1_loss(pred.squeeze(), labels.squeeze())

def zinc_metric(pred, labels, batch_size):
    return F.l1_loss(pred.squeeze(), labels.squeeze()).item() * batch_size

In [ ]:
from dgl.data import ZINCDataset

zinc_data = make_graph_data_bundle(
    ZINCDataset(mode='train'),
    ZINCDataset(mode='valid'),
    ZINCDataset(mode='test'),
    batch_size=128,
    drop_last_train=True,
)


In [ ]:
zinc_experiment_specs = [
    ('pna_edge', ZINC_PNA_EDGE, 'best_pna_model.pth'),
    ('pna_noedge', ZINC_PNA_NOEDGE, 'best_pna_noedge_model.pth'),
    ('pna_no_scalers_edge', ZINC_PNA_NOSCALER_EDGE, 'best_pna_noscalers_model.pth'),
    ('pna_no_scalers_noedge', ZINC_PNA_NOSCALER_NOEDGE, 'best_pna_noscalers_noedge_model.pth'),
    ('mpnn_sum_edge', ZINC_MPNN_SUM_EDGE, 'best_mpnn_sum_model.pth'),
    ('mpnn_sum_noedge', ZINC_MPNN_SUM_NOEDGE, 'best_mpnn_sum_noedge_model.pth'),
    ('mpnn_max_edge', ZINC_MPNN_MAX_EDGE, 'best_mpnn_max_model.pth'),
    ('mpnn_max_noedge', ZINC_MPNN_MAX_NOEDGE, 'best_mpnn_max_noedge_model.pth'),
]

zinc_results = {}
for experiment_name, config_factory, checkpoint_path in zinc_experiment_specs:
    zinc_results[experiment_name] = run_benchmark_experiment(
        config_factory=config_factory,
        data_bundle=zinc_data,
        preprocess_fn=preprocess_zinc,
        loss_fn=zinc_loss,
        metric_fn=zinc_metric,
        device=device,
        checkpoint_path=checkpoint_path,
        experiment_name=f'ZINC/{experiment_name}',
    )


In [ ]:
class BaseCIFAR10Config(BaseConfig):
    name = 'CIFAR10'
    task_dim = 10
    node_dim = 5
    scheduler_mode = 'max'

# --- 2.1 CIFAR10 PNA ---
class CIFAR10_PNA_NOEDGE(BaseCIFAR10Config):
    aggregators = ['mean', 'max', 'min', 'std']
    scalers = ['identity', 'amplification', 'attenuation']
    edge_feat = False
    edge_dim = 0
    hidden_dim = 75
    out_dim = 70
    dropout = 0.1
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5

class CIFAR10_PNA_EDGE(BaseCIFAR10Config):
    aggregators = ['mean', 'max', 'min', 'std']
    scalers = ['identity', 'amplification', 'attenuation']
    edge_feat = True
    edge_dim = 50
    hidden_dim = 75
    out_dim = 70
    dropout = 0.3
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5

# --- 2.2 CIFAR10 PNA (No Scalers) ---
class CIFAR10_PNA_NOSCALER_NOEDGE(BaseCIFAR10Config):
    aggregators = ['mean', 'max', 'min', 'std']
    scalers = ['identity']
    edge_feat = False
    edge_dim = 0
    hidden_dim = 95
    out_dim = 90
    dropout = 0.1
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5

class CIFAR10_PNA_NOSCALER_EDGE(BaseCIFAR10Config):
    aggregators = ['mean', 'max', 'min', 'std']
    scalers = ['identity']
    edge_feat = True
    edge_dim = 50
    hidden_dim = 95
    out_dim = 90
    dropout = 0.3
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5

# --- 2.3 CIFAR10 MPNN (Sum) ---
class CIFAR10_MPNN_SUM_NOEDGE(BaseCIFAR10Config):
    aggregators = ['sum']
    scalers = ['identity']
    edge_feat = False
    edge_dim = 0
    hidden_dim = 110
    out_dim = 90
    dropout = 0.2
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5
    weight_decay = 3e-6

class CIFAR10_MPNN_SUM_EDGE(BaseCIFAR10Config):
    aggregators = ['sum']
    scalers = ['identity']
    edge_feat = True
    edge_dim = 20
    hidden_dim = 110
    out_dim = 90
    dropout = 0.2
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5

# --- 2.4 CIFAR10 MPNN (Max) ---
class CIFAR10_MPNN_MAX_NOEDGE(BaseCIFAR10Config):
    aggregators = ['max']
    scalers = ['identity']
    edge_feat = False
    edge_dim = 0
    hidden_dim = 110
    out_dim = 90
    dropout = 0.2
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5
    weight_decay = 3e-6

class CIFAR10_MPNN_MAX_EDGE(BaseCIFAR10Config):
    aggregators = ['max']
    scalers = ['identity']
    edge_feat = True
    edge_dim = 20
    hidden_dim = 110
    out_dim = 90
    dropout = 0.2
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5



class BaseMNISTConfig(BaseConfig):
    name = 'MNIST'
    task_dim = 10
    node_dim = 3
    scheduler_mode = 'max'

# --- 3.1 MNIST PNA ---
class MNIST_PNA_NOEDGE(BaseMNISTConfig):
    aggregators = ['mean', 'max', 'min', 'std']
    scalers = ['identity', 'amplification', 'attenuation']
    edge_feat = False
    edge_dim = 0
    hidden_dim = 75
    out_dim = 70
    dropout = 0.1
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5

class MNIST_PNA_EDGE(BaseMNISTConfig):
    aggregators = ['mean', 'max', 'min', 'std']
    scalers = ['identity', 'amplification', 'attenuation']
    edge_feat = True
    edge_dim = 50
    hidden_dim = 75
    out_dim = 70
    dropout = 0.3
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5

# --- 3.2 MNIST PNA (No Scalers) ---
class MNIST_PNA_NOSCALER_NOEDGE(BaseMNISTConfig):
    aggregators = ['mean', 'max', 'min', 'std']
    scalers = ['identity']
    edge_feat = False
    edge_dim = 0
    hidden_dim = 95
    out_dim = 90
    dropout = 0.1
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5

class MNIST_PNA_NOSCALER_EDGE(BaseMNISTConfig):
    aggregators = ['mean', 'max', 'min', 'std']
    scalers = ['identity']
    edge_feat = True
    edge_dim = 50
    hidden_dim = 95
    out_dim = 90
    dropout = 0.3
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5

# --- 3.3 MNIST MPNN (Sum) ---
class MNIST_MPNN_SUM_NOEDGE(BaseMNISTConfig):
    aggregators = ['sum']
    scalers = ['identity']
    edge_feat = False
    edge_dim = 0
    hidden_dim = 110
    out_dim = 90
    dropout = 0.2
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5
    weight_decay = 3e-6

class MNIST_MPNN_SUM_EDGE(BaseMNISTConfig):
    aggregators = ['sum']
    scalers = ['identity']
    edge_feat = True
    edge_dim = 20
    hidden_dim = 110
    out_dim = 90
    dropout = 0.2
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5

# --- 3.4 MNIST MPNN (Max) ---
class MNIST_MPNN_MAX_NOEDGE(BaseMNISTConfig):
    aggregators = ['max']
    scalers = ['identity']
    edge_feat = False
    edge_dim = 0
    hidden_dim = 110
    out_dim = 90
    dropout = 0.2
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5
    weight_decay = 3e-6

class MNIST_MPNN_MAX_EDGE(BaseMNISTConfig):
    aggregators = ['max']
    scalers = ['identity']
    edge_feat = True
    edge_dim = 20
    hidden_dim = 110
    out_dim = 90
    dropout = 0.2
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5


def preprocess_image(batched_graph, labels, device, use_edge_feat=True):
    batched_graph = batched_graph.to(device)
    labels = labels.to(device).long()
    h = batched_graph.ndata['feat'].float()
    if use_edge_feat and batched_graph.num_edges() > 0:
        e = batched_graph.edata['feat'].float()
        if e.dim() == 1:
            e = e.unsqueeze(-1)
    else:
        e = None
    return batched_graph, h, e, labels

def image_loss(pred, labels):
    return F.cross_entropy(pred, labels)

def image_metric(pred, labels, batch_size):
    preds = pred.argmax(dim=1)
    correct = (preds == labels).sum().item()
    return correct

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /root/.dgl/Superpixel_c7da4aa2
!unzip -q /content/drive/MyDrive/Superpixel.zip -d /root/.dgl/Superpixel_c7da4aa2/

!mv /root/.dgl/Superpixel_c7da4aa2/Superpixel_c7da4aa2/* /root/.dgl/Superpixel_c7da4aa2/

!rm -rf /root/.dgl/Superpixel_c7da4aa2/__MACOSX
!rm -rf /root/.dgl/Superpixel_c7da4aa2/Superpixel_c7da4aa2


!ls /root/.dgl/Superpixel_c7da4aa2/

In [ ]:
from dgl.data import CIFAR10SuperPixelDataset

cifar10_data = make_superpixel_data_bundle(
    CIFAR10SuperPixelDataset,
    train_size=45000,
    batch_size=128,
    seed=42,
    drop_last_train=True,
)


In [ ]:
cifar10_experiment_specs = [
    ('pna_noedge', CIFAR10_PNA_NOEDGE, 'best_pna_noedge_cifar10_model.pth', None),
    ('pna_no_scalers_noedge', CIFAR10_PNA_NOSCALER_NOEDGE, 'best_pna_noscaler_noedge_cifar10_model.pth', None),
    ('mpnn_sum_noedge', CIFAR10_MPNN_SUM_NOEDGE, 'best_mpnn_sum_noedge_cifar10_model.pth', 40),
    ('mpnn_sum_edge', CIFAR10_MPNN_SUM_EDGE, 'best_mpnn_sum_cifar10_model.pth', 40),
]

cifar10_results = {}
for experiment_name, config_factory, checkpoint_path, epoch_limit in cifar10_experiment_specs:
    cifar10_results[experiment_name] = run_benchmark_experiment(
        config_factory=config_factory,
        data_bundle=cifar10_data,
        preprocess_fn=preprocess_image,
        loss_fn=image_loss,
        metric_fn=image_metric,
        device=device,
        checkpoint_path=checkpoint_path,
        experiment_name=f'CIFAR10/{experiment_name}',
        epochs=epoch_limit,
    )


In [ ]:
from dgl.data import MNISTSuperPixelDataset

mnist_data = make_superpixel_data_bundle(
    MNISTSuperPixelDataset,
    train_size=55000,
    batch_size=128,
    seed=42,
    drop_last_train=True,
)


In [ ]:
mnist_experiment_specs = [
    ('pna_edge', MNIST_PNA_EDGE, 'best_pna_mnist_model.pth', None),
    ('pna_no_scalers_noedge', MNIST_PNA_NOSCALER_NOEDGE, 'best_pna_mnist_noscalers_noedge_model.pth', None),
    ('mpnn_sum_noedge', MNIST_MPNN_SUM_NOEDGE, 'best_mpnn_mnist_sum_noedge_model.pth', 40),
    ('mpnn_sum_edge', MNIST_MPNN_SUM_EDGE, 'best_mpnn_mnist_sum_model.pth', 40),
]

mnist_results = {}
for experiment_name, config_factory, checkpoint_path, epoch_limit in mnist_experiment_specs:
    mnist_results[experiment_name] = run_benchmark_experiment(
        config_factory=config_factory,
        data_bundle=mnist_data,
        preprocess_fn=preprocess_image,
        loss_fn=image_loss,
        metric_fn=image_metric,
        device=device,
        checkpoint_path=checkpoint_path,
        experiment_name=f'MNIST/{experiment_name}',
        epochs=epoch_limit,
    )


In [ ]:
from google.colab import drive
from torch.utils.data import Subset
from ogb.graphproppred import DglGraphPropPredDataset

drive.mount('/content/drive')
!mkdir -p /content/dataset
!unzip -q /content/drive/MyDrive/ogbg_molhiv.zip -d /content/dataset/
!ls /content/dataset/


molhiv_data = make_ogb_data_bundle(
    dataset_name='ogbg-molhiv',
    root='/content/dataset/',
    batch_size=128,
)

print(
    f"MolHIV splits: train={len(molhiv_data['datasets']['train'])}, "
    f"valid={len(molhiv_data['datasets']['valid'])}, "
    f"test={len(molhiv_data['datasets']['test'])}"
)


In [ ]:
class BaseConfig_MOLHIV:
    name = 'MolHIV'
    epochs = 200
    batch_size = 128
    init_lr = 0.01
    lr_reduce_factor = 0.5
    min_lr = 1e-4
    weight_decay = 3e-6
    lr_schedule_patience = 20
    scheduler_mode = 'max'
    num_layers = 4
    task_dim = 1
    dropout = 0.3
    in_feat_dropout = 0.0
    use_batch_norm = True
    use_graph_norm = True
    readout = 'mean'


    edge_feat = False
    edge_dim = 0
    towers = 1
    divide_input_first = False
    divide_input_last = False
    pretrans_layers = 1
    posttrans_layers = 1



class Config_MOLHIV_PNA(BaseConfig_MOLHIV):
    aggregators = ['mean', 'max', 'min', 'std']
    scalers = ['identity', 'amplification', 'attenuation']
    hidden_dim = 70
    out_dim = 70


class Config_MOLHIV_PNA_NOSCALER(BaseConfig_MOLHIV):
    aggregators = ['mean', 'max', 'min', 'std']
    scalers = ['identity']
    hidden_dim = 95
    out_dim = 95


class Config_MOLHIV_MPNN_SUM(BaseConfig_MOLHIV):
    aggregators = ['sum']
    scalers = ['identity']
    hidden_dim = 120
    out_dim = 120


class Config_MOLHIV_MPNN_MAX(BaseConfig_MOLHIV):
    aggregators = ['max']
    scalers = ['identity']
    hidden_dim = 120
    out_dim = 120


In [ ]:
def molhiv_epoch(model, dataloader, device, evaluator, config,
                 optimizer=None):
    """Run one MolHIV train or evaluation epoch."""
    is_training = optimizer is not None
    model.train(is_training)
    total_loss = 0.0
    labels_all, predictions_all = [], []

    with torch.set_grad_enabled(is_training):
        for batched_graph, labels in dataloader:
            batched_graph = batched_graph.to(device)
            labels = labels.to(device).float()
            node_features = batched_graph.ndata['feat']
            edge_features = batched_graph.edata['feat'] if config.edge_feat else None

            node_counts = batched_graph.batch_num_nodes()
            graph_norm = torch.cat([
                torch.ones(count, 1) / math.sqrt(count)
                for count in node_counts
            ]).to(device)
            predictions = model(
                batched_graph, node_features, graph_norm, edge_features
            )
            labeled_mask = ~torch.isnan(labels)
            loss = F.binary_cross_entropy_with_logits(
                predictions[labeled_mask], labels[labeled_mask]
            )

            if is_training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * batched_graph.batch_size
            labels_all.append(labels.detach().cpu())
            predictions_all.append(predictions.detach().cpu())

    labels_array = torch.cat(labels_all, dim=0).numpy()
    predictions_array = torch.cat(predictions_all, dim=0).numpy()
    roc_auc = evaluator.eval({
        'y_true': labels_array,
        'y_pred': predictions_array,
    })['rocauc']
    return total_loss / len(dataloader.dataset), roc_auc


def run_molhiv_experiment(config_factory, data_bundle, device,
                          checkpoint_path, experiment_name):
    """Train one MolHIV model and retain the checkpoint with best validation ROC-AUC."""
    experiment_config = config_factory()
    experiment_model = RealWorldBenchmarkModel(
        experiment_config, data_bundle['avg_log_d']
    ).to(device)
    experiment_optimizer = optim.Adam(
        experiment_model.parameters(),
        lr=experiment_config.init_lr,
        weight_decay=experiment_config.weight_decay,
    )
    experiment_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        experiment_optimizer,
        mode=experiment_config.scheduler_mode,
        factor=experiment_config.lr_reduce_factor,
        patience=experiment_config.lr_schedule_patience,
    )
    evaluator = Evaluator(name='ogbg-molhiv')
    best_val_roc_auc = float('-inf')
    best_test_roc_auc = float('nan')
    loaders = data_bundle['loaders']

    for epoch_index in range(experiment_config.epochs):
        train_loss, train_roc_auc = molhiv_epoch(
            experiment_model, loaders['train'], device, evaluator,
            experiment_config, optimizer=experiment_optimizer,
        )
        val_loss, val_roc_auc = molhiv_epoch(
            experiment_model, loaders['valid'], device, evaluator,
            experiment_config,
        )
        experiment_scheduler.step(val_roc_auc)
        current_lr = experiment_optimizer.param_groups[0]['lr']
        test_text = ''

        if val_roc_auc > best_val_roc_auc:
            best_val_roc_auc = val_roc_auc
            _, best_test_roc_auc = molhiv_epoch(
                experiment_model, loaders['test'], device, evaluator,
                experiment_config,
            )
            torch.save(experiment_model.state_dict(), checkpoint_path)
            test_text = f' | Test ROC-AUC: {best_test_roc_auc:.4f} (New Best!)'

        print(
            f'{experiment_name} | Epoch {epoch_index:03d} | '
            f'LR: {current_lr:.6f} | Train Loss: {train_loss:.4f} | '
            f'Val ROC-AUC: {val_roc_auc:.4f}{test_text}'
        )
        if current_lr < experiment_config.min_lr:
            break

    print(f'{experiment_name} | Best validation ROC-AUC: '
          f'{best_val_roc_auc:.4f} | Test at best validation: '
          f'{best_test_roc_auc:.4f}')
    return {
        'best_val_roc_auc': best_val_roc_auc,
        'test_roc_auc_at_best_val': best_test_roc_auc,
        'checkpoint_path': checkpoint_path,
    }


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    dgl.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(41)

In [ ]:
molhiv_experiment_specs = [
    ('pna', Config_MOLHIV_PNA, 'best_pna_molhiv_model.pth'),
    ('pna_no_scalers', Config_MOLHIV_PNA_NOSCALER, 'best_pna_molhi_noscaler_model.pth'),
    ('mpnn_sum', Config_MOLHIV_MPNN_SUM, 'best_mpnn_molhiv_sum_model.pth'),
    ('mpnn_max', Config_MOLHIV_MPNN_MAX, 'best_mpnn_molhiv_max_model.pth'),
]

molhiv_results = {}
for experiment_name, config_factory, checkpoint_path in molhiv_experiment_specs:
    molhiv_results[experiment_name] = run_molhiv_experiment(
        config_factory=config_factory,
        data_bundle=molhiv_data,
        device=device,
        checkpoint_path=checkpoint_path,
        experiment_name=f'MolHIV/{experiment_name}',
    )
